In [1]:
import _referAsMain

import math
import time
import random
import sys
import numpy
from IPython.display import SVG, display
from holo.pointers import Pointer
from holo.profilers import Profiler
from holo.prettyFormats import prettyPrint, prettyTime

print(sys.version_info)

import torch
from torch.utils.data import DataLoader

from paths_cfg import TOKENIZER_SAVE_DIRECTORY, OUR_DATASET_DIRECTORY
from dataset import svg_dataset
import tokenizer_pfe.tokenizer_project as tokenizerLib
import metrics.metrics

from LLM.model import Model, GenerationStats
import LLM.nanochat.gpt as nanoChatModel
from LLM.nanochat.common import compute_init, autodetect_device_type

added '/home/holo/dev/PFE_LLM_art_generation' to import paths
sys.version_info(major=3, minor=13, micro=12, releaselevel='final', serial=0)


In [2]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

Torch version: 2.9.1+cu128
CUDA available: True
CUDA version: 12.8
Device count: 1


In [3]:
try:
    torch.cuda.empty_cache()
    del model  # type: ignore
    torch.cuda.empty_cache()
except Exception:
    pass
model = Model.load("model_5.5M_1K", versionID=16, device=torch.device("cuda:0"))
model.show_infos()

loading the tokenizer from: /home/holo/dev/PFE_LLM_art_generation/models_save/model_5.5M_1K/tokenizer.json
loading the historique from: /home/holo/dev/PFE_LLM_art_generation/models_save/model_5.5M_1K/versions/version_0016_checkpoint-14/history.json
Padding vocab_size from 585 to 640 for efficiency
Scaling the LR for the AdamW parameters ∝1/√(256/768) = 1.732051
GPTConfig(sequence_len=8192, vocab_size=585, n_layer=6, n_head=1, n_kv_head=1, n_embd=256, window_pattern='SSSL')
5_537_900 total params (embeding: 655_360 | last layer: 163_840 | transformer: 4_718_688)
on device: device(type='cuda', index=0), with effective context_size: 4096
-> 129.958 MFLOPs/token 
-> 532.31 GFLOPs with full context


In [4]:
print(f"trained for {model.nb_epoches_done} epoches")
for k, v in model.historique.get_all_historique().items():
    print(f"  {k}: {v.get(model.nb_epoches_done-1, None):.4g}")  # type: ignore

trained for 14 epoches
  CE_train: 0.5459
  CE2_train: 0.4841
  PPL_train: 1.726
  PPL2_train: 1.623
  BPC_train: 0.4016
  ENTROPY_train: 0.5453
  LOGITS_SD_train: 2.892
  TOP-1_train: 0.8637
  TOP-5_train: 0.8991
  epoch_duration: 2187


In [5]:
try:
    torch.cuda.empty_cache()
    del model  # type: ignore
    torch.cuda.empty_cache()
except Exception:
    pass
model = Model.load("model_5.5M_1K", versionID=22, device=torch.device("cuda:0"))
model.show_infos()

loading the tokenizer from: /home/holo/dev/PFE_LLM_art_generation/models_save/model_5.5M_1K/tokenizer.json
loading the historique from: /home/holo/dev/PFE_LLM_art_generation/models_save/model_5.5M_1K/versions/version_0022_checkpoint-20/history.json
Padding vocab_size from 585 to 640 for efficiency
Scaling the LR for the AdamW parameters ∝1/√(256/768) = 1.732051
GPTConfig(sequence_len=8192, vocab_size=585, n_layer=6, n_head=1, n_kv_head=1, n_embd=256, window_pattern='SSSL')
5_537_900 total params (embeding: 655_360 | last layer: 163_840 | transformer: 4_718_688)
on device: device(type='cuda', index=0), with effective context_size: 4096
-> 129.958 MFLOPs/token 
-> 532.31 GFLOPs with full context


In [6]:
print(f"trained for {model.nb_epoches_done} epoches")
for k, v in model.historique.get_all_historique().items():
    print(f"  {k}: {v.get(model.nb_epoches_done-1, None):.4g}")  # type: ignore

trained for 20 epoches
  CE_train: 0.5428
  CE2_train: 0.4813
  PPL_train: 1.721
  PPL2_train: 1.618
  BPC_train: 0.3993
  ENTROPY_train: 0.5422
  LOGITS_SD_train: 2.849
  TOP-1_train: 0.8643
  TOP-5_train: 0.8998
  epoch_duration: 2178
